<a href="https://colab.research.google.com/github/Chris-p05/NPL-Project/blob/main/OptiHireDataScraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# --- STEP 1: GLOBAL SETUP & LIBRARIES ---
!pip install torch tensorflow transformers nltk spacy pymupdf scrapingbee kaggle -q
import os, json, base64, requests, pymupdf, spacy, pandas as pd
from google.colab import userdata

In [4]:
# --- STEP 2: SECURE CREDENTIALS --- (SET THEM UP ON THE SECRETS TAB)
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [5]:
# --- STEP 3: DATA ACQUISITION (THE "GOOD" DATASET) ---
!kaggle datasets download -d snehaanbhawal/resume-dataset
!unzip -o resume-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset
License(s): CC0-1.0
100% 62.5M/62.5M [00:00<00:00, 275MB/s]

Archive:  resume-dataset.zip
  inflating: Resume/Resume.csv       
  inflating: data/data/ACCOUNTANT/10554236.pdf  
  inflating: data/data/ACCOUNTANT/10674770.pdf  
  inflating: data/data/ACCOUNTANT/11163645.pdf  
  inflating: data/data/ACCOUNTANT/11759079.pdf  
  inflating: data/data/ACCOUNTANT/12065211.pdf  
  inflating: data/data/ACCOUNTANT/12202337.pdf  
  inflating: data/data/ACCOUNTANT/12338274.pdf  
  inflating: data/data/ACCOUNTANT/12442909.pdf  
  inflating: data/data/ACCOUNTANT/12780508.pdf  
  inflating: data/data/ACCOUNTANT/12802330.pdf  
  inflating: data/data/ACCOUNTANT/13072019.pdf  
  inflating: data/data/ACCOUNTANT/13130984.pdf  
  inflating: data/data/ACCOUNTANT/13294301.pdf  
  inflating: data/data/ACCOUNTANT/13491889.pdf  
  inflating: data/data/ACCOUNTANT/13701259.pdf  
  inflating: data/data/ACCOUNTANT/14055988.pdf  
  inflating: d

In [6]:
# --- STEP 4: PROFESSIONAL LOGIC ENGINE ---
try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    spacy.cli.download("en_core_web_md")
    nlp = spacy.load("en_core_web_md")

def clean_and_anonymize(text):
    """
    Combines PII redaction and text normalization into one high-tier pass.

    """
    if not isinstance(text, str): return ""

    # 1. Anonymize to eliminate human bias
    doc = nlp(text)
    redacted_text = text
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "GPE", "DATE"]:
            redacted_text = redacted_text.replace(ent.text, f"[{ent.label_}_REDACTED]")

    # 2. Normalize for Semantic Analysis
    clean_doc = nlp(redacted_text.lower())
    tokens = [t.lemma_ for t in clean_doc if not t.is_stop and not t.is_punct]
    return " ".join(tokens)

def get_base64_resume(file_path):
    with open(file_path, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
# --- STEP 5: BATCH PROCESSING (KAGGLE DATA) --- THIS IS WHERE WE TEST A SPECIFIC RESUME AGAINST THE DATASET
# We use 'Resume.csv' and 'Resume_str' from the new dataset.
try:
    df = pd.read_csv('Resume/Resume.csv') # Adjust path if unzip structure differs
except:
    df = pd.read_csv('Resume.csv')

processed_data = []
print("Processing the REAL resume text now!")

# Processing 50 samples to test the logic
for index, row in df.head(50).iterrows():
    processed_data.append({
        "id": row['ID'],
        "category": row['Category'],
        "clean_resume": clean_and_anonymize(row['Resume_str'])
    })

Processing the REAL resume text now!


In [8]:
# --- STEP 6: SCRAPINGBEE & INDIVIDUAL EVALUATION --- (AGAIN, SET THEM UP IN SECRETS TAB)
# Connects your live job data to a specific file upload.
try:
    client = ScrapingBeeClient(api_key=userdata.get('SCRAPINGBEE_KEY'))

    response = client.get(
        'https://www.simplyhired.com/search?q=python+developer',
        params={'ai_query': 'Extract job title, company, requirements, and responsibilities', 'render_js': True}
    )
    job_data = response.json()
except Exception as e:
    print(f"ScrapingBee Error: {e}. Check your API key!")

# Evaluate "my_resume.pdf" if the user managed to upload it.
pdf_file = "my_resume.pdf"
if os.path.exists(pdf_file):
    encoded_resume = get_base64_resume(pdf_file)
    print(f"Ready to evaluate {pdf_file} against {job_data.get('job_title', 'Job Posting')}")
else:
    print(f"Warning: {pdf_file} not found. Upload it to the folder!")

print(f"\nPhase 1 & 2 COMPLETE. {len(processed_data)} resumes processed with zero bias.")

ScrapingBee Error: name 'ScrapingBeeClient' is not defined. Check your API key!

Phase 1 & 2 COMPLETE. 50 resumes processed with zero bias.


In [9]:
# --- STEP 7: SEMANTIC MATCHING ENGINE (WEEK 2 GOAL) ---
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Load the SBERT model (Optimized for performance)
# all-MiniLM-L6-v2 is fast and perfect for resume matching.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Move model to GPU if available (which we confirmed in step 1)
if torch.cuda.is_available():
    model = model.to('cuda')

def get_match_score(resume_text, job_description_text):
    """
    Calculates the semantic similarity between a resume and a job description.
    Returns a score from 0-100%.
    """
    # Encode both texts into semantic vectors (embeddings)
    resume_emb = model.encode(resume_text, convert_to_tensor=True)
    job_emb = model.encode(job_description_text, convert_to_tensor=True)

    # Compute Cosine Similarity (0 to 1)
    cosine_score = util.cos_sim(resume_emb, job_emb)

    # Convert to 0-100% scale as planned in our slides
    return round(float(cosine_score) * 100, 2)

# --- EXAMPLE TEST ---
# Testing our cleaned resumes against a sample job requirement
sample_jd = "Looking for Python developer with database experience"
clean_jd = clean_and_anonymize(sample_jd)

print(f"Testing against JD: '{sample_jd}'\n")
for resume in processed_data[:5]: # Testing the first 5 from our processed batch
    score = get_match_score(resume['clean_resume'], clean_jd)
    print(f"Resume ID {resume['id']} | Category: {resume['category']} | Match Score: {score}%")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Testing against JD: 'Looking for Python developer with database experience'

Resume ID 16852973 | Category: HR | Match Score: 8.45%
Resume ID 22323967 | Category: HR | Match Score: 13.09%
Resume ID 33176873 | Category: HR | Match Score: 14.03%
Resume ID 27018550 | Category: HR | Match Score: 16.18%
Resume ID 17812897 | Category: HR | Match Score: 10.72%


In [10]:
# --- STEP 8: MODEL REFINEMENT - KEYWORD CLUSTERING & WEIGHTING (WEEK 3) ---

def extract_keywords(text):
    """
    Uses spaCy to extract technical keywords and noun chunks.
    """
    doc = nlp(text.lower())
    # Extracting nouns and proper nouns as 'keywords'
    keywords = set([token.text for token in doc if token.pos_ in ["NOUN", "PROPN"] and not token.is_stop])
    return keywords

def refined_match_score(resume_text, job_description_text):
    # 1. Get Base Semantic Score (from our Step 7 model)
    semantic_score = get_match_score(resume_text, job_description_text)

    # 2. Calculate Keyword Alignment
    jd_keywords = extract_keywords(job_description_text)
    resume_keywords = extract_keywords(resume_text)

    if not jd_keywords: return semantic_score

    # Calculate how many JD keywords appear in the resume
    intersection = jd_keywords.intersection(resume_keywords)
    keyword_score = (len(intersection) / len(jd_keywords)) * 100

    # 3. Weighted Final Score (70% Semantic + 30% Keyword Match)
    final_score = (0.7 * semantic_score) + (0.3 * keyword_score)
    return round(final_score, 2)

# --- TEST THE REFINEMENT ---
print(f"Refining evaluation for the top candidate...")
top_resume = processed_data[0]['clean_resume']
test_jd = "Looking for a Python developer with experience in SQL, PyTorch, and NLP."

legacy_score = get_match_score(top_resume, test_jd)
new_score = refined_match_score(top_resume, test_jd)

print(f"Original SBERT Score: {legacy_score}%")
print(f"Refined Weighted Score: {new_score}%")

Refining evaluation for the top candidate...
Original SBERT Score: 12.25%
Refined Weighted Score: 13.57%
